# Этап 3 — эксперименты по улучшению качества

Каждый эксперимент меняет **ровно одну** вещь относительно бейзлайна Этапа 2
(`deeplabv3plus_r50-d8_1xb16-practice_dataset-256x256.py`, test mDice = 90.10),
чтобы прирост можно было приписать конкретному изменению.

| # | что меняем | конфиг |
|---|---|---|
| 1 | **усиленные аугментации**: дефолтный PhotoMetric + RandomRotFlip + RandomCutOut | [`exp1_augs_strong.py`](../configs/deeplabv3plus_practice/exp1_augs_strong.py) |
| 2 | **лосс CE:Dice = 1:3** (вместо 1:1) | [`exp2_loss_ce1dice3.py`](../configs/deeplabv3plus_practice/exp2_loss_ce1dice3.py) |
| 3 | **backbone R101-d16-mg124** (вместо R50-d8) | [`exp3_r101_d16_mg124.py`](../configs/deeplabv3plus_practice/exp3_r101_d16_mg124.py) |

**Предварительные условия:** окружение и ClearML уже настроены в Этапе 2 (см.
[`train.ipynb`](train.ipynb)). Здесь только запуск экспериментов и итоговый
сравнительный анализ на test.

Каждый эксперимент тренируется ~30–45 мин на T4 (по таблице замеров скорости из
урока «Современные модификации»). Итого 3 эксперимента ≈ 1.5–2 ч GPU.

In [ ]:
# cwd ядра ставим в корень проекта (VSCode открывает с cwd = папка ноутбука)
%cd ~/nn_cv_sprint2_full

## 1. Эксперимент 1 — усиленные аугментации

ClearML task: `exp1_augs_strong` (создастся при запуске).

In [ ]:
!python3.10 tools/train.py configs/deeplabv3plus_practice/exp1_augs_strong.py

## 2. Эксперимент 2 — лосс CE:Dice = 1:3

ClearML task: `exp2_loss_ce1_dice3`.

In [ ]:
!python3.10 tools/train.py configs/deeplabv3plus_practice/exp2_loss_ce1dice3.py

## 3. Эксперимент 3 — backbone R101-d16-mg124

ClearML task: `exp3_r101_d16_mg124`. По таблице замеров из урока этот вариант
не только сильнее, но и **быстрее** R50-d8.

In [ ]:
!python3.10 tools/train.py configs/deeplabv3plus_practice/exp3_r101_d16_mg124.py

## 4. Сравнение на test — baseline + 3 эксперимента

Для каждого конфига:
1. находим лучший чекпойнт (`best_mDice_epoch_*.pth`);
2. прогоняем `tools/test.py` — это даёт каноничную mmseg-метрику (mDice / mIoU / per-class);
3. прогоняем `per_image_dice.py` — сохраняет 5 best/worst PNG-триплетов в `supplementary/viz/test_<tag>/`.

Финальную таблицу с числами впишем в README Этапа 3 руками — числа берём из строк
вида `aAcc: ... mDice: ... mIoU: ...` в выводе ниже.

In [ ]:
import glob

EXPS = [
    ("baseline",     "configs/deeplabv3plus_practice/deeplabv3plus_r50-d8_1xb16-practice_dataset-256x256.py"),
    ("exp1_augs",    "configs/deeplabv3plus_practice/exp1_augs_strong.py"),
    ("exp2_loss13",  "configs/deeplabv3plus_practice/exp2_loss_ce1dice3.py"),
    ("exp3_r101mg",  "configs/deeplabv3plus_practice/exp3_r101_d16_mg124.py"),
]

for tag, cfg in EXPS:
    name = cfg.split('/')[-1].replace('.py','')
    ckpts = sorted(glob.glob(f'work_dirs/{name}/best_mDice_epoch_*.pth'))
    if not ckpts:
        print(f'!! no best ckpt for {tag} (work_dirs/{name}/), skip'); continue
    ckpt = ckpts[-1]
    print(f'\n========== {tag} ({ckpt}) ==========')
    print('--- tools/test.py (mmseg canonical mDice) ---')
    !python3.10 tools/test.py {cfg} {ckpt}
    print('\n--- per_image_dice (saves best/worst PNG) ---')
    !python3.10 practicum_work/src/analysis/per_image_dice.py \
        --config {cfg} --checkpoint {ckpt} --split test \
        --out practicum_work/supplementary/viz/test_{tag} --n 5